# Task 06 — Semantic Search for CoreTech Services
**Intern:** Muhammad Taha  
**Company:** CoreTech Innovations (coretechio.com)  
**Internship Track:** AI Engineering  
**Description:** Build a semantic search system over the CoreTech knowledge base using TF-IDF vectorization and Cosine Similarity to return the top 3 most relevant results for any user query.

## Step 1 — Install and Import Libraries

In [ ]:
# Install required libraries
# scikit-learn : TF-IDF vectorization and Cosine Similarity
# pandas       : Load and manage the CSV knowledge base
# numpy        : Numerical operations on similarity score arrays
!pip install scikit-learn pandas numpy --quiet

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries imported successfully.')
print(f'  numpy   version : {np.__version__}')
print(f'  pandas  version : {pd.__version__}')

## Step 2 — Load and Explore the Knowledge Base

In [ ]:
# Load the CoreTech knowledge base CSV into a Pandas DataFrame
df = pd.read_csv('coretech_knowledge_base.csv')

print(f'Knowledge Base Loaded Successfully')
print(f'Total Records : {len(df)}')
print(f'Columns       : {list(df.columns)}')
print()

# Show first 5 records
print('First 5 Records:')
df.head()

In [ ]:
# Explore the dataset — category distribution using pandas value_counts
print('Records by Category:')
category_dist = df['category'].value_counts()
print(category_dist.to_string())

# Use numpy to compute text length statistics
text_lengths = np.array([len(str(t)) for t in df['text']])
print(f'\nText Length Statistics:')
print(f'  Average : {np.mean(text_lengths):.0f} characters')
print(f'  Maximum : {np.max(text_lengths)} characters')
print(f'  Minimum : {np.min(text_lengths)} characters')
print(f'  Std Dev : {np.std(text_lengths):.0f} characters')

## Step 3 — Build the TF-IDF Matrix

In [ ]:
# Initialize TF-IDF Vectorizer
# TF-IDF (Term Frequency-Inverse Document Frequency) converts text into
# numerical vectors. Words frequent in one document but rare across others
# get higher scores — making them better discriminators for search.
#
# Parameters:
#   stop_words='english'  : Remove common English words (the, is, and...)
#   ngram_range=(1,2)     : Capture single words AND 2-word phrases
#   max_df=0.95           : Ignore terms in more than 95% of documents
#   min_df=1              : Include terms appearing in at least 1 document

vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_df=0.95,
    min_df=1
)

# Fit and transform the corpus into a TF-IDF matrix
# Shape: (number of records, number of unique terms/ngrams)
tfidf_matrix = vectorizer.fit_transform(df['text'].tolist())

print(f'TF-IDF Matrix Built Successfully')
print(f'  Matrix Shape     : {tfidf_matrix.shape}')
print(f'  Records          : {tfidf_matrix.shape[0]}')
print(f'  Unique Terms     : {tfidf_matrix.shape[1]}')
print(f'  Matrix Type      : {type(tfidf_matrix).__name__} (sparse)')

# Show top 20 most important terms by IDF score using numpy
feature_names = vectorizer.get_feature_names_out()
idf_scores    = vectorizer.idf_
top_indices   = np.argsort(idf_scores)[::-1][:20]
print(f'\nTop 20 Most Distinctive Terms (by IDF score):')
for idx in top_indices:
    print(f'  {feature_names[idx]:<35} IDF: {idf_scores[idx]:.4f}')

## Step 4 — Define Semantic Search Function

In [ ]:
def semantic_search(query: str, top_n: int = 3) -> pd.DataFrame:
    """
    Search the CoreTech knowledge base for the most relevant records.

    Steps:
        1. Transform query using the fitted TF-IDF vectorizer
        2. Compute Cosine Similarity between query and all records
        3. Rank by score (descending) using numpy argsort
        4. Return top N results as a DataFrame

    Cosine Similarity:
        Measures the cosine of the angle between two TF-IDF vectors.
        Score of 1.0 = identical direction (perfect match)
        Score of 0.0 = orthogonal vectors (no shared terms)

    Args:
        query  (str) : User search query
        top_n  (int) : Number of results to return (default: 3)

    Returns:
        pd.DataFrame : Top N results with similarity scores
    """
    # Step 1: Transform query into TF-IDF vector
    query_vector = vectorizer.transform([query])

    # Step 2: Compute Cosine Similarity between query and all documents
    # Result shape: (1, n_records)
    scores = cosine_similarity(query_vector, tfidf_matrix)

    # Step 3: Flatten to 1D numpy array and get top N indices
    scores_flat = scores.flatten()
    top_indices = np.argsort(scores_flat)[::-1][:top_n]

    # Step 4: Build results DataFrame
    results = df.iloc[top_indices].copy()
    results['similarity_score'] = np.round(scores_flat[top_indices], 4)
    results = results.reset_index(drop=True)

    return results[['id', 'category', 'title', 'similarity_score', 'text']]


def display_results(query: str, results: pd.DataFrame) -> None:
    """Print search results in a clean readable format."""
    print(f'\n{"─" * 65}')
    print(f'  Query: "{query}"')
    print(f'{"─" * 65}')

    if results.empty or results['similarity_score'].max() == 0:
        print('  No relevant results found. Try rephrasing your query.')
        return

    for i, row in results.iterrows():
        score = row['similarity_score']
        bar   = '█' * int(np.round(score * 10))
        level = 'High' if score >= 0.3 else 'Medium' if score >= 0.1 else 'Low'
        print(f'\n  #{i+1}  Score: {score:.4f}  [{bar:<10}]  Relevance: {level}')
        print(f'  Category : {row["category"]}')
        print(f'  Title    : {row["title"]}')
        preview = str(row['text'])[:180] + ('...' if len(str(row['text'])) > 180 else '')
        print(f'  Preview  : {preview}')
    print(f'{"─" * 65}')

print('Semantic search function defined successfully.')

## Step 5 — Run Test Queries (Top 3 Results with Scores)

In [ ]:
# Test Case 1: Web Development
query = 'web development services for enterprise'
results = semantic_search(query, top_n=3)
display_results(query, results)
print('\nRaw Results DataFrame:')
results[['category', 'title', 'similarity_score']]

In [ ]:
# Test Case 2: Internship Application
query = 'how to apply for internship program'
results = semantic_search(query, top_n=3)
display_results(query, results)
results[['category', 'title', 'similarity_score']]

In [ ]:
# Test Case 3: Cybersecurity
query = 'cybersecurity and data protection services'
results = semantic_search(query, top_n=3)
display_results(query, results)
results[['category', 'title', 'similarity_score']]

In [ ]:
# Test Case 4: Cloud Modernization
query = 'cloud modernization and legacy migration'
results = semantic_search(query, top_n=3)
display_results(query, results)
results[['category', 'title', 'similarity_score']]

In [ ]:
# Test Case 5: Company founders and leadership
query = 'founders CEO leadership team coretech'
results = semantic_search(query, top_n=3)
display_results(query, results)
results[['category', 'title', 'similarity_score']]

In [ ]:
# Test Case 6: ERP Systems
query = 'ERP systems for manufacturing and healthcare'
results = semantic_search(query, top_n=3)
display_results(query, results)
results[['category', 'title', 'similarity_score']]

In [ ]:
# Test Case 7: Contact and Location
query = 'contact email phone location Pakistan'
results = semantic_search(query, top_n=3)
display_results(query, results)
results[['category', 'title', 'similarity_score']]

In [ ]:
# Test Case 8: Client retention stats
query = 'client retention rate and company statistics'
results = semantic_search(query, top_n=3)
display_results(query, results)
results[['category', 'title', 'similarity_score']]

## Step 6 — Similarity Score Analysis with Pandas and NumPy

In [ ]:
# Compute similarity scores for all records against a test query
# and analyse the score distribution using pandas and numpy

test_query   = 'software development services'
query_vector = vectorizer.transform([test_query])
all_scores   = cosine_similarity(query_vector, tfidf_matrix).flatten()

# Build analysis DataFrame using pandas
analysis_df = df[['id', 'category', 'title']].copy()
analysis_df['similarity_score'] = np.round(all_scores, 4)
analysis_df = analysis_df.sort_values('similarity_score', ascending=False)

print(f'Score Distribution for query: "{test_query}"')
print(f'\nAll records ranked by similarity score:')
print(analysis_df[['category', 'title', 'similarity_score']].to_string(index=False))

print(f'\nNumPy Score Statistics:')
print(f'  Mean Score  : {np.mean(all_scores):.4f}')
print(f'  Max Score   : {np.max(all_scores):.4f}')
print(f'  Min Score   : {np.min(all_scores):.4f}')
print(f'  Std Dev     : {np.std(all_scores):.4f}')
print(f'  Records > 0.1 threshold : {np.sum(all_scores > 0.1)}')

---
## Summary

| Component | Details |
|---|---|
| Dataset | coretech_knowledge_base.csv — 35 records across 5 categories |
| Categories | Company Info, Services, Projects, Internship, FAQ |
| Vectorization | TF-IDF with unigrams and bigrams, English stop words removed |
| Similarity Metric | Cosine Similarity (scikit-learn) |
| Results Returned | Top 3 matches with similarity scores |
| Libraries | pandas, numpy, scikit-learn |
| Search Queries Tested | 8 test cases covering all knowledge base categories |